<a href="https://colab.research.google.com/github/Gurram-Bhaskar/planetary-rover-navigation/blob/main/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Planetary Rover GRPO Training (Colab)

This notebook runs the same GRPO pipeline as `train.py` for the OpenEnv hackathon submission.

Pipeline:
- Llama 3.2 1B + Unsloth 4-bit NF4 + LoRA
- TRL `GRPOTrainer`
- Two-tier rewards: format gatekeeper + environment reward from FastAPI physics server

## 1) Prepare Runtime

In Colab, place this repo at `/content/Scaler_meta_hugging` (clone or upload) before running the next cell.

In [2]:
import os
import pathlib
import subprocess

# Define the target directory
repo_path = pathlib.Path('/content/planetary-rover-navigation')

# Clone the repo if it doesn't exist
if not repo_path.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/Gurram-Bhaskar/planetary-rover-navigation.git", str(repo_path)], check=True)
else:
    print("Repository already exists.")

# Change into the directory
os.chdir(repo_path)
print('Working directory:', os.getcwd())

Cloning repository...
Working directory: /content/planetary-rover-navigation


In [3]:
%pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.0/457.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.3/390.3 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7

In [4]:
import requests
import subprocess
import sys
import time

server_proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'
])

deadline = time.time() + 30
server_ready = False
while time.time() < deadline:
    try:
        res = requests.get('http://127.0.0.1:7860/tasks', timeout=2)
        if res.ok:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    raise RuntimeError('Server failed to start within 30 seconds.')

print('FastAPI environment server is live on :7860')

FastAPI environment server is live on :7860


## 2) Run GRPO Training

This cell imports `train.py`, sets runtime knobs, and starts training.
Look for `METRICS {'loss': ..., 'reward': ...}` in output for your submission screenshot.

In [5]:
import os
import train as rover_train

os.environ['ROVER_SERVER_URL'] = 'http://127.0.0.1:7860'
os.environ['ROVER_NUM_GENERATIONS'] = '8'

rover_train.SERVER_URL = os.environ['ROVER_SERVER_URL']
rover_train.NUM_GENERATIONS = int(os.environ['ROVER_NUM_GENERATIONS'])

rover_train.main()

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('Stopped FastAPI environment server.')
else:
    print('No running server process found.')